# Eniac A/B-Test: Welcher Button bringt die meisten Klicks?

**Ziel:** Vier Versionen des Buttons auf der Eniac-Startseite vergleichen und prüfen, ob die Unterschiede in der Klickrate (CTR) statistisch signifikant sind.

| Version | Farbe | Text |
|---|---|---|
| A (Original) | weiß | JETZT SHOPPEN |
| B | rot | JETZT SHOPPEN |
| C | weiß | ANGEBOTE ANSEHEN |
| D | rot | ANGEBOTE ANSEHEN |

**Methode:** Klickrate berechnen, Chi-Quadrat-Test über alle Versionen, danach Post-hoc-Tests mit Bonferroni-Korrektur.

## 1. Bibliotheken importieren

In [ ]:
import os
import itertools
import pandas as pd
from scipy import stats

## 2. Daten laden
Die vier CSV-Dateien liegen im Ordner `data/`. In Colab kannst du hier stattdessen den Pfad zu deinem Drive-Ordner eintragen.

In [ ]:
folder = "data/"   # In Colab mit Drive: "/content/drive/MyDrive/DEIN_ORDNERNAME/"

df_a = pd.read_csv(folder + "eniac_a.csv")   # Version A: weiß, SHOP NOW
df_b = pd.read_csv(folder + "eniac_b.csv")   # Version B: rot, SHOP NOW
df_c = pd.read_csv(folder + "eniac_c.csv")   # Version C: weiß, SEE DEALS
df_d = pd.read_csv(folder + "eniac_d.csv")   # Version D: rot, SEE DEALS

df_a.head()

## 3. Klicks und Besuche pro Version auslesen
Die Besuche stehen in der Zeile `mySidebar`, die Klicks in der Zeile des Buttons.

In [ ]:
versions = {"A": df_a, "B": df_b, "C": df_c, "D": df_d}
results = []

for name, df in versions.items():
    # Besuche: Zahl vor "visits" aus dem Snapshot-Text herausschneiden
    snapshot = df.loc[df["Name"] == "mySidebar", "Snapshot information"].iloc[0]
    visits = int(snapshot.split("•")[2].split("visits")[0])

    # Klicks auf den Button (SHOP NOW bzw. SEE DEALS)
    clicks = df.loc[df["Name"].isin(["SHOP NOW", "SEE DEALS"]), "No. clicks"].iloc[0]

    results.append({"version": name, "clicks": clicks, "visits": visits})

results = pd.DataFrame(results)
results

## 4. Klickrate (CTR) berechnen
Klickrate = Klicks / Besuche

In [ ]:
results["ctr_percent"] = results["clicks"] / results["visits"] * 100
results

In [ ]:
# Balkendiagramm der Klickrate speichern
os.makedirs("images", exist_ok=True)

ax = results.set_index("version")["ctr_percent"].plot.bar(
    title="Klickrate (CTR) pro Version", ylabel="CTR in %", rot=0
)
ax.figure.savefig("images/ctr_per_version.png", bbox_inches="tight")

**Beobachtung:** Die roten Versionen (B, D) schneiden deutlich schlechter ab als die weißen (A, C). Version C liegt rechnerisch vorne, der Abstand zu A ist aber klein. Ob die Unterschiede Zufall sind, prüfen wir jetzt.

## 5. Chi-Quadrat-Test über alle Versionen

- **Nullhypothese:** Alle vier Versionen haben die gleiche Klickrate, Unterschiede sind Zufall.
- **Alternativhypothese:** Mindestens eine Version unterscheidet sich.
- **Signifikanzniveau:** α = 0,05

In [ ]:
# Kein Klick = Besuche minus Klicks
results["no_clicks"] = results["visits"] - results["clicks"]

# Kontingenztabelle: Zeilen = Klick / Kein Klick, Spalten = Versionen
contingency_table = results.set_index("version")[["clicks", "no_clicks"]].T
contingency_table

In [ ]:
chisq, pvalue, dof, expected = stats.chi2_contingency(contingency_table)
print("p-Wert:", pvalue)
print("Signifikant:", pvalue < 0.05)

Der p-Wert ist praktisch null und damit kleiner als 0,05. Wir lehnen die Nullhypothese ab: Die Klicks hängen von der Version ab. Welche Versionen sich unterscheiden, zeigt der Post-hoc-Test.

## 6. Post-hoc-Tests mit Bonferroni-Korrektur

Jedes der 6 Paare wird einzeln getestet. Weil mehrere Tests das Risiko für einen Fehler 1. Art erhöhen, wird α durch die Anzahl der Vergleiche geteilt: 0,05 / 6 ≈ 0,0083.

In [ ]:
pairs = list(itertools.combinations(contingency_table.columns, 2))

alpha = 0.05
alpha_post_hoc = alpha / len(pairs)
print("Neues Alpha:", round(alpha_post_hoc, 4))

for a, b in pairs:
    chisq, pvalue, dof, expected = stats.chi2_contingency(contingency_table[[a, b]])
    print(a, "vs.", b, "| p-Wert:", pvalue, "| signifikant:", pvalue < alpha_post_hoc)

| Paar | Ergebnis |
|---|---|
| A vs. B | signifikant (B schlechter) |
| A vs. C | **nicht signifikant** |
| A vs. D | signifikant (D schlechter) |
| B vs. C | signifikant |
| B vs. D | signifikant |
| C vs. D | signifikant |

**Fazit Klickrate:** Weiß ist besser als Rot. Zwischen A und C gibt es keinen nachweisbaren Unterschied. Es gibt also noch keinen eindeutigen Gewinner.

## 7. Zusätzliche Kennzahlen: Abbruchrate und Rückkehrrate

Das Website-Team hat zusätzlich die Abbruchrate der verlinkten Seite und die Rückkehrrate zur Startseite erfasst (niedriger ist jeweils besser). Für Version B fehlen die Daten wegen eines Fehlers in der Datenerfassung.

*Die Werte sind ungefähr aus den Diagrammen der Kursunterlagen abgelesen.*

In [ ]:
extra = pd.DataFrame(
    {"drop_off_rate": [62, 71, 69.5], "homepage_return_rate": [5.3, 4.7, 2.6]},
    index=["A", "C", "D"],
)
extra

In [ ]:
ax = extra["drop_off_rate"].plot.bar(title="Abbruchrate (ca.)", ylabel="in %", rot=0)
ax.figure.savefig("images/drop_off_rate.png", bbox_inches="tight")

In [ ]:
ax = extra["homepage_return_rate"].plot.bar(title="Rückkehrrate zur Startseite (ca.)", ylabel="in %", rot=0)
ax.figure.savefig("images/homepage_return_rate.png", bbox_inches="tight")

## 8. Fazit und Empfehlung

- **D und B scheiden aus:** Die roten Buttons haben eine signifikant schlechtere Klickrate. Für B fehlen zudem die Zusatzdaten.
- **A oder C:** Bei der Klickrate nicht unterscheidbar. A hat die klar niedrigere Abbruchrate (ca. 62 % gegen 71 %), C nur eine etwas niedrigere Rückkehrrate.
- **Empfehlung:** Version A (das Original) behalten. Für einen belegten Wechsel zu C sollte ein weiteres Experiment nur mit A und C laufen.

**Einschränkungen:** Der MDE von 20 % war für den Test eingeplant, der Unterschied zwischen A und C liegt mit etwa 5 % deutlich darunter. Die Daten von Version A tragen ein anderes Erstellungsdatum (14.09.2021) als die von B, C und D (27.10.2021), was den Vergleich verfälschen könnte.